# Training "hey Amy"

Trimmed from openWakeWord's `automatic_model_training.ipynb` to what this wake word needs.

**Before running: Runtime → Change runtime type → T4 GPU.**

Removed from the original: the TensorFlow/tflite toolchain, which only exists to produce a `.tflite`
alongside the ONNX. The daemon loads ONNX, and those three pins (`tensorflow-cpu==2.8.1`,
`tensorflow_probability==0.16.0`, `onnx_tf==1.10.0`) date from 2022 and are the most likely thing in
the notebook to fail to install today.

Expect roughly 45–60 minutes, most of it generating speech.

## 1 · Install

In [ ]:
# piper-sample-generator, which speaks the training clips
!git clone -q https://github.com/rhasspy/piper-sample-generator
!wget -q -O piper-sample-generator/models/en_US-libritts_r-medium.pt \
  'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt'
!pip install -q piper-phonemize webrtcvad deep-phonemizer==0.0.19

# openwakeword, from source so the training script is available
!git clone -q https://github.com/dscripka/openwakeword
!pip install -q -e ./openwakeword

# training dependencies: augmentation, metrics, and the phonetic dictionary used to
# invent near-misses of the target phrase
!pip install -q mutagen==1.47.0 torchinfo==1.8.0 torchmetrics==1.2.0 speechbrain==0.5.14 \
  audiomentations==0.33.0 torch-audiomentations==0.11.0 acoustics==0.2.6 \
  pronouncing==0.2.0 datasets==2.14.6

# the frozen feature models the classifier sits on top of
import os
os.makedirs("./openwakeword/openwakeword/resources/models", exist_ok=True)
base = "https://github.com/dscripka/openWakeWord/releases/download/v0.5.1"
!wget -q {base}/embedding_model.onnx -O ./openwakeword/openwakeword/resources/models/embedding_model.onnx
!wget -q {base}/melspectrogram.onnx -O ./openwakeword/openwakeword/resources/models/melspectrogram.onnx
print("installed")

In [ ]:
import os, sys, numpy as np, torch, yaml, scipy, datasets
from pathlib import Path
from tqdm import tqdm

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "NONE — set Runtime > Change runtime type > T4 GPU, then rerun from the top")

## 2 · Download the negatives

Three things: room impulse responses to make clean speech sound like a room, background noise and
music to mix under it, and precomputed embeddings of ~2000 hours of speech that the classifier
learns to say no to.

In [ ]:
# Room impulse responses (MIT survey)
os.makedirs("./mit_rirs", exist_ok=True)
rir = datasets.load_dataset("davidscripka/MIT_environmental_impulse_responses", split="train", streaming=True)
for row in tqdm(rir, desc="RIRs"):
    scipy.io.wavfile.write(os.path.join("./mit_rirs", row['audio']['path'].split('/')[-1]),
                           16000, (row['audio']['array'] * 32767).astype(np.int16))

In [ ]:
# Background noise (one AudioSet shard) and music (Free Music Archive)
os.makedirs("audioset", exist_ok=True)
!wget -q -O audioset/bal_train09.tar https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/bal_train09.tar
!cd audioset && tar -xf bal_train09.tar

os.makedirs("./audioset_16k", exist_ok=True)
ds = datasets.Dataset.from_dict({"audio": [str(i) for i in Path("audioset/audio").glob("**/*.flac")]})
ds = ds.cast_column("audio", datasets.Audio(sampling_rate=16000))
for row in tqdm(ds, desc="noise"):
    name = row['audio']['path'].split('/')[-1].replace(".flac", ".wav")
    scipy.io.wavfile.write(os.path.join("./audioset_16k", name), 16000,
                           (row['audio']['array'] * 32767).astype(np.int16))

os.makedirs("./fma", exist_ok=True)
fma = iter(datasets.load_dataset("rudraml/fma", name="small", split="train", streaming=True)
           .cast_column("audio", datasets.Audio(sampling_rate=16000)))
N_HOURS = 1     # clips are 30s; raise for a stronger model at the cost of download time
for _ in tqdm(range(N_HOURS * 3600 // 30), desc="music"):
    row = next(fma)
    name = row['audio']['path'].split('/')[-1].replace(".mp3", ".wav")
    scipy.io.wavfile.write(os.path.join("./fma", name), 16000,
                           (row['audio']['array'] * 32767).astype(np.int16))

In [ ]:
# Precomputed speech embeddings: ~2000 hours of negatives, and an 11 hour validation set
# used to measure false alarms per hour.
!wget -q --show-progress https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy
!wget -q --show-progress https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy

## 3 · Configure for "hey Amy"

The phrase settings come from the config in the dsh-lite repo; the paths stay as downloaded above.

In [ ]:
import urllib.request

config = yaml.load(open("openwakeword/examples/custom_model.yml").read(), yaml.Loader)

amy = yaml.safe_load(urllib.request.urlopen(
    "https://raw.githubusercontent.com/shaunbeach/dsh-lite/main/voice/training/hey_amy.yaml").read())

# Only the keys that describe the phrase. feature_data_files, batch_n_per_class and every path
# describe what was just downloaded and where it landed, and belong to this notebook.
for key in ("target_phrase", "model_name", "custom_negative_phrases",
            "n_samples", "n_samples_val", "steps",
            "max_negative_weight", "target_false_positives_per_hour"):
    config[key] = amy[key]

config["background_paths"] = ["./audioset_16k", "./fma"]
config["false_positive_validation_data_path"] = "validation_set_features.npy"
config["feature_data_files"] = {"ACAV100M_sample": "openwakeword_features_ACAV100M_2000_hrs_16bit.npy"}
config["output_dir"] = "./hey_amy_training"

with open("hey_amy.yaml", "w") as f:
    yaml.dump(config, f)

print(config["target_phrase"], "->", config["model_name"])
print(len(config["custom_negative_phrases"]), "custom negatives,",
      config["n_samples"], "positives,", config["steps"], "training steps")

## 4 · Generate, augment, train

The first cell is the long one. If it is interrupted, run it again: it counts what already exists
and carries on rather than starting over.

In [ ]:
# Speak the phrase, and the near-misses, thousands of times
!{sys.executable} openwakeword/openwakeword/train.py --training_config hey_amy.yaml --generate_clips

In [ ]:
# Put those clips in rooms and under noise
!{sys.executable} openwakeword/openwakeword/train.py --training_config hey_amy.yaml --augment_clips

In [ ]:
# Train. The tflite conversion at the very end fails without TensorFlow, which is expected and
# harmless: the ONNX is written before it, and ONNX is what the daemon loads.
!{sys.executable} openwakeword/openwakeword/train.py --training_config hey_amy.yaml --train_model || true

## 5 · Check it, then download

In [ ]:
import onnxruntime, numpy as np

model_path = "hey_amy_training/hey_amy.onnx"
assert os.path.exists(model_path), "no model was produced — check the training output above"
print(f"{model_path}  {os.path.getsize(model_path)/1024:.0f} KB")

# It should load and score an embedding-shaped input without complaint.
sess = onnxruntime.InferenceSession(model_path, providers=["CPUExecutionProvider"])
name = sess.get_inputs()[0].name
print("input :", name, sess.get_inputs()[0].shape)
print("silence scores:", float(sess.run(None, {name: np.zeros((1, 16, 96), dtype=np.float32)})[0][0][0]))

In [ ]:
from google.colab import files
files.download("hey_amy_training/hey_amy.onnx")

## On your Mac

```sh
mkdir -p ~/.dsh/wake
mv ~/Downloads/hey_amy.onnx ~/.dsh/wake/
dsh-voice --wake-word ~/.dsh/wake/hey_amy.onnx
```

It should print `Amy is listening. Say "hey amy" to wake her.`

If it fires when nobody spoke, raise `--wake-threshold` above 0.5. If it misses you, lower it and
watch the scores with `dsh-voice -v`.